In [28]:
import pandas as pd
import numpy as np

In [29]:
# read data - exclude timestamp to save memory
df = pd.read_csv(
    "../ml-latest/ratings.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
    },
    usecols=["userId", "movieId", "rating"]
)

df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,110,4.0
2,1,158,4.0
3,1,260,4.5
4,1,356,5.0


In [30]:
# there shouldn't be any null values but just check anyway - delete if there are any
print(f"Any null values for userId: {df['userId'].isnull().any()}")
print(f"Any null values for movieId: {df['movieId'].isnull().any()}")
print(f"Any null values for rating: {df['rating'].isnull().any()}")

Any null values for userId: False
Any null values for movieId: False
Any null values for rating: False


In [31]:
# are there any ratings below 0
df[df['rating'] < 0]

,userId,movieId,rating


In [32]:
print(f"Min rating value: {df["rating"].min()}")
print(f"Max rating value: {df["rating"].max()}")

Min rating value: 0.5
Max rating value: 5.0


In [33]:
'''
Creating confidence column
confidence = 1 + alpha * rating
can also use confidence = rating (works fine), but confidence is more industry standard
'''
alpha = 10
df['confidence'] = (1 + alpha * df['rating']).astype(np.float32)
df.head()

,userId,movieId,rating,confidence
0,1,1,4.0,41.0
1,1,110,4.0,41.0
2,1,158,4.0,41.0
3,1,260,4.5,46.0
4,1,356,5.0,51.0


In [34]:
# filter sparse users/items
# users with 1 rating -> meaningless embeddings
# movies with 1 rating -> random noise
# model overfits noise
# memory wasted on useless rows/cols

min_user_interactions = 5
min_movie_interactions = 5

df = df.groupby("userId").filter(lambda x: len(x) >= min_user_interactions)
df = df.groupby("movieId").filter(lambda x: len(x) >= min_movie_interactions)

In [35]:
# encode ids into contiguous integers
df["userId"], user_index = pd.factorize(df["userId"])
df["userId"] = df["userId"].astype(np.int32)
                                  
df["movieId"], movie_index = pd.factorize(df["movieId"])
df["movieId"] = df["movieId"].astype(np.int32)
df.head()

,userId,movieId,rating,confidence
0,0,0,4.0,41.0
1,0,1,4.0,41.0
2,0,2,4.0,41.0
3,0,3,4.5,46.0
4,0,4,5.0,51.0


In [36]:
type(user_index)

pandas.Index